In [1]:
import json
from langchain_core.documents import Document

## 1. Creating LangChain Documents

In [2]:
def load_rules(path):
    """Load the rule cards created in notebook 01."""
    with open(path) as f:
        return json.load(f)

def rules_to_documents(rules):
    """Turn each rule card into a LangChain Document."""
    output = []
    for rule in rules:
        output.append(
            
            Document(
                page_content=rule["text"],
                metadata={
                    "rule_id": rule["rule_id"],
                    "title": rule["title"],
                    "updated": rule["updated"],
                    "chapter": rule["chapter"],
                    "start_page": rule["start_page"],
                    "end_page": rule["end_page"],
                    "printed_start": rule["printed_start"],
                    "printed_end": rule["printed_end"],
                }
            )

        )

    return output

rules = load_rules("../data/processed/b3_rules.json")
documents = rules_to_documents(rules)

print(len(documents))
print(documents[0].metadata)
print(documents[0].page_content[:200])

123
{'rule_id': 'B3-1-01', 'title': 'Comprehensive Risk Assessment', 'updated': '11/05/2025', 'chapter': 'B3-1', 'start_page': 286, 'end_page': 288, 'printed_start': 268, 'printed_end': 269}
B3-1-01, Comprehensive Risk Assessment (11/05/2025)
Introduction
This topic contains information on the comprehensive risk assessment approach to underwriting, including:
Overview
Comprehensive Risk A


## 2. Chunking

In [9]:
from collections import Counter
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1500, chunk_overlap=200):
    """Split long documents into chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(documents)

chunks = split_documents(documents)

print("Total chunks:", len(chunks))

per_rule = Counter(c.metadata["rule_id"] for c in chunks)
print("Rules kept whole:", sum(1 for n in per_rule.values() if n == 1))
print("Most-split rules:", per_rule.most_common(3))

Total chunks: 401
Rules kept whole: 34
Most-split rules: [('B3-5.3-09', 18), ('B3-2-02', 15), ('B3-6-05', 13)]


In [8]:
dti_chunks = [c for c in chunks if c.metadata["rule_id"] == "B3-6-02"]
for i, c in enumerate(dti_chunks):
    print(f"\n--- Chunk {i} ({len(c.page_content)} chars) ---")
    print(c.page_content[:150])


--- Chunk 0 (1478 chars) ---
B3-6-02, Debt-to-Income Ratios (04/02/2025)
Introduction
This topic contains information on the use of the debt-to-income (DTI) ratio, including:
DTI 

--- Chunk 1 (1482 chars) ---
including:
cash-out refinance transactions — the maximum ratio may be lower for loan casefiles underwritten
through DU (see B2-1.3-03, Cash-Out Refina

--- Chunk 2 (1488 chars) ---
subordinate lien payments) or rental payments (see B3-6-05, Monthly Debt Obligations);
if the subject loan is a second home or investment property, us

--- Chunk 3 (1416 chars) ---
when qualifying borrowers. This is acceptable as long as Fannie Mae’s minimum requirements are met,
and lenders consistently apply the same approach t

--- Chunk 4 (1451 chars) ---
In all cases, if the lender determines that there is new subordinate financing on the subject property during the
loan process, the mortgage loan must

--- Chunk 5 (1462 chars) ---
recalculated outside of DU.
4
• If the recalculated DTI ratio exc